In [54]:
# Import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', 50)

# Set path to datasets
DATA_DIR = Path("/home/tella/code/StellaRodriguesLallement/OSE_Project/Dataset_visualization/new_data_folder/data/extracted_datasets")

# Check it exists
print(DATA_DIR.exists())


True


In [55]:
# LOADING DATASETS

df_financial = pd.read_csv(DATA_DIR / '02_financial_data.csv', dtype={'siren': str})
df_kpi       = pd.read_csv(DATA_DIR / '07_kpi_data.csv', dtype={'siren': str})

In [56]:
df_financial.shape, df_kpi.shape

((18116, 12), (18116, 28))

In [57]:
df_financial.columns.tolist()

['company_name',
 'siren',
 'siret',
 'caBilan',
 'caConsolide',
 'caGroupe',
 'dateConsolide',
 'fondsPropres',
 'resultatExploitation',
 'resultatNet',
 'trancheCaBilan',
 'trancheCaConsolide']

In [58]:
df_kpi.columns.tolist()

['company_name',
 'siren',
 'siret',
 'year',
 'fonds_propres',
 'ca_france',
 'date_cloture_exercice',
 'duree_exercice',
 'salaires_traitements',
 'charges_financieres',
 'impots_taxes',
 'ca_bilan',
 'resultat_exploitation',
 'dotations_amortissements',
 'capital_social',
 'code_confidentialite',
 'resultat_bilan',
 'annee',
 'effectif',
 'effectif_sous_traitance',
 'filiales_participations',
 'evolution_ca',
 'subventions_investissements',
 'ca_export',
 'evolution_effectif',
 'participation_bilan',
 'ca_consolide',
 'resultat_net_consolide']

In [59]:
df_fin_feat = df_financial.copy()
df_kpi_feat= df_kpi.copy()


In [61]:
df_fin_feat.isnull().mean().sort_values(ascending=False)

caConsolide             0.978527
caGroupe                0.978527
trancheCaConsolide      0.975326
resultatExploitation    0.507342
fondsPropres            0.405884
resultatNet             0.380824
caBilan                 0.326893
trancheCaBilan          0.323526
dateConsolide           0.230459
siret                   0.025723
company_name            0.000000
siren                   0.000000
dtype: float64

In [63]:
df_fin_feat = df_fin_feat.drop(columns=['caConsolide','caGroupe','trancheCaConsolide'])

In [64]:
df_kpi_feat.isnull().mean().sort_values(ascending=False)

capital_social                 1.000000
dotations_amortissements       1.000000
participation_bilan            1.000000
evolution_effectif             1.000000
ca_export                      1.000000
subventions_investissements    1.000000
evolution_ca                   1.000000
filiales_participations        1.000000
effectif_sous_traitance        1.000000
code_confidentialite           1.000000
resultat_net_consolide         1.000000
salaires_traitements           1.000000
duree_exercice                 1.000000
impots_taxes                   1.000000
charges_financieres            1.000000
ca_france                      1.000000
ca_consolide                   0.978527
resultat_exploitation          0.507342
fonds_propres                  0.405884
resultat_bilan                 0.380824
ca_bilan                       0.326893
effectif                       0.322588
date_cloture_exercice          0.230459
annee                          0.230459
year                           0.230459


In [67]:
# create a list of columns to drop
cols_to_drop = df_kpi_feat.columns[df_kpi_feat.isna().mean() > 0.95]
df_kpi_feat = df_kpi_feat.drop(columns=cols_to_drop)

df_kpi_feat.columns

Index(['company_name', 'siren', 'siret', 'year', 'fonds_propres',
       'date_cloture_exercice', 'ca_bilan', 'resultat_exploitation',
       'resultat_bilan', 'annee', 'effectif'],
      dtype='object')

In [69]:
df_fin_feat.shape, df_kpi_feat.shape

((18116, 9), (18116, 11))

In [71]:
df_fin_feat['dateConsolide'] = pd.to_datetime(df_fin_feat['dateConsolide'], errors='coerce')
# df_fin_feat['dateConsolide'] = df_financial['dateConsolide'].dt.tz_localize(None)
df_fin_feat['dateConsolide'] = df_fin_feat['dateConsolide'].dt.year


In [74]:
df_fin_feat['dateConsolide'].value_counts().sort_values(ascending=False)


dateConsolide
2023.0    4203
2022.0    1316
2021.0    1034
2024.0     515
2020.0     422
2019.0     207
2018.0     196
2017.0     170
1970.0     152
2016.0     115
2015.0      52
2014.0      48
2013.0      47
2012.0      37
2011.0      31
2010.0      27
2009.0      17
2008.0      16
2025.0      14
2007.0      11
2006.0       8
2003.0       2
Name: count, dtype: int64

In [ ]:
df_financial.head(5)

In [ ]:
df_financial['siren'].nunique()


In [ ]:
df_financial.shape[0]


In [ ]:
df_financial['siren'].value_counts().head(20)


In [ ]:
(df_financial
 .sort_values(['siren', 'dateConsolide'])
 .groupby('siren')['dateConsolide']
 .nunique()
 .value_counts()
 .head(10))


In [ ]:
siren_multidate= (df_financial.groupby('siren')['dateConsolide'].nunique()
        .loc[lambda x: x == 2]
        .index
)


In [ ]:
df_financial[df_financial['siren'].isin(siren_multidate)].sort_values('siren').head(50)

In [ ]:
df_financial.shape

In [ ]:
df_financial = df_financial.drop_duplicates()
df_financial.shape

In [ ]:
dups = df_financial[df_financial['siren'].isin(siren_multidate)]

# Compare row 0 and row 1 of each duplicated siren:
dups.groupby('siren').apply(lambda g: g.nunique())


In [ ]:
def compare_rows(g):
    if len(g) <= 1:
        return None
    return (g.iloc[0] != g.iloc[1])

dups.groupby('siren').apply(compare_rows)


In [ ]:
(df_financial
 .sort_values(['siren', 'dateConsolide'])
 .groupby('siren')['dateConsolide']
 .nunique()
 .value_counts()
 .head(10))

In [ ]:
df_financial['dateConsolide'] = pd.to_datetime(df_financial['dateConsolide'], errors='coerce')
df_financial['dateConsolide'] = df_financial['dateConsolide'].dt.tz_localize(None)
df_financial.head(2)


In [ ]:
df_financial['dateConsolide'] = df_financial['dateConsolide'].dt.year
df_financial['dateConsolide'].value_counts().sort_values(by='dateConsolide', ascending=False)

In [ ]:
df_financial.isnull().sum()

In [ ]:
df_financial_recent = df_financial[df_financial['annee'] >= 2023]


In [ ]:
financial_feat_selected = [
    'siren',
    'annee',

    # Solvency
    'fonds_propres',
    'capital_social',
    'resultat_bilan',
    'charges_financieres',

    # Profitability
    'resultat_exploitation',

    # Revenue & activity
    'ca_france',
    'ca_export',
    'evolution_ca',

    # Workforce
    'effectif',
    'evolution_effectif',

    # Useful operational metadata
    'date_cloture_exercice',
]
